In [19]:
import pandas as pd
import os
import seaborn as sns
import matplotlib.pyplot as plt
from NSWData.NSWDataLoader import *
from ModelFiles.ModelPlots import *
from ModelFiles.ModelConfigs import HORIZONS
nsw_data_loader = NSWDataLoader()

Found NSW data path: C:\Users\kyim1\Desktop\capstone_project_GroupA\data\NSW


## Selecting the Best SARIMAX Model

In [6]:
sarimax_results_df = pd.read_csv(os.path.join(nsw_data_loader.output_dir, 'sarimax_master_validation_fitting_log.csv'))
sarimax_results_df = sarimax_results_df[['model_name','order','seasonal_order','time_taken_seconds','training_size','validation_horizon','validation_rmse']]

In [25]:
sarimax_results_df['training_size'].value_counts()

training_size
720      9
1440     9
2880     9
5760     9
11520    3
Name: count, dtype: int64

We have in total, 4 different (excl. 11520 as it was too big to run for higher order settings) training sizes (AKA input timesteps from the last timestep in training data).

We will iterate over below to find the best model:
* Retrieve best training_size for each model (order/seasonal_order)
* Compare models to retrieve the best model

In [48]:
best_sarimax_models_df = pd.DataFrame()
for horizon in HORIZONS:
    df = sarimax_results_df[sarimax_results_df['validation_horizon'] == horizon].copy()
    best_train_size_5 =  df[(df['order'] == '(5, 0, 0)') & (df['seasonal_order'] == '(1, 1, 1, 48)')].sort_values('validation_rmse').iloc[0]
    best_train_size_4 = df[(df['order'] == '(4, 0, 0)') & (df['seasonal_order'] == '(1, 1, 1, 48)')].sort_values('validation_rmse').iloc[0]
    best_train_size_3 = df[(df['order'] == '(3, 0, 0)') & (df['seasonal_order'] == '(1, 1, 1, 48)')].sort_values('validation_rmse').iloc[0]
    df = pd.concat([best_train_size_5, best_train_size_4, best_train_size_3], axis=1).T
    best_sarimax_models_df = pd.concat([best_sarimax_models_df, df], axis=0)
best_sarimax_models_df['rank_val_rmse'] = best_sarimax_models_df.groupby('validation_horizon')['validation_rmse'].rank().astype(int)
best_sarimax_models_df = best_sarimax_models_df.sort_values(['validation_horizon', 'rank_val_rmse'])
best_sarimax_models_df[['time_taken_seconds','validation_rmse']] = best_sarimax_models_df[['time_taken_seconds','validation_rmse']].apply(lambda x: round(x, 2))
best_sarimax_models_df.to_csv('best_sarimax_models.csv', index=False)

Observations:
* the optimal training size is 1440 timesteps, which is 30 days.
* for horizon 48, non-seasonal autoregressive order = 5 had the best RMSE, but took the longest to fit.
* for horizons 336 and 720, non-seasonal autoregressive order = 4 has the best RMSE.

Despite performing the worst for horizon 48, the RMSE of model sarimax_20260417_172703 (the one with AR order 4) is 504, which is not too far away than the best RMSE of 492. The model also fits around 17% faster as well.

**Thus, to keep things simple, we will select order (4,0,0) and seasonal order (1,1,1,48) as our best model.**

## Selecting the Best LSTM Model

In [46]:
lstm_results_df = pd.read_csv(os.path.join(nsw_data_loader.output_dir, 'lstm_master_validation_fitting_log.csv'))
lstm_results_df = lstm_results_df[(lstm_results_df['seed'] == 31415) & (lstm_results_df['best_epoch'] == lstm_results_df['epoch'])]
lstm_results_df = lstm_results_df[['file_name','horizon','lookback','hidden_size','num_layers','dropout','num_attention_heads','val_loss']]

In [47]:
lstm_results_df

,file_name,horizon,lookback,hidden_size,num_layers,dropout,num_attention_heads,val_loss
60,biLSTM_20260417_144435_fitting_log.csv,48,168,128,2,0.20,4,0.114284
96,biLSTM_20260417_150303_fitting_log.csv,48,168,256,4,0.35,8,0.125065
193,biLSTM_20260417_155420_fitting_log.csv,336,336,128,2,0.20,4,0.254338
219,biLSTM_20260417_163408_fitting_log.csv,336,336,256,4,0.35,8,0.301078
248,biLSTM_20260417_174216_fitting_log.csv,720,720,128,2,0.20,4,0.344234
272,biLSTM_20260417_180503_fitting_log.csv,720,720,256,4,0.35,8,0.350059


Due to the lengthened time to train LSTM models on our hardware, we only compare 2 models:
1) "Large" with hidden_size=256, num_layers=4, num_attention_heads=8 and dropout=35%
2) "Small" with hidden_size=128, num_layers=2, num_attention_heads=4 and dropout=20%

We can already clearly see the "Small" model outperforms the "Large".